# 🎭 OmniVoice V8 — Documentation-Correct Emotional TTS

**`docs/expressive-speech.md` (Jul 20, 2026) অনুযায়ী সঠিক approach:**

| ✅ সঠিক | ❌ ভুল |
|---------|--------|
| প্রতিটা beat = আলাদা API call | পুরো স্ক্রিপ্ট এক call-এ |
| `[laughter]`, `[sigh]`, `[pause]`, `[breath]` | `[cry]`, `[whisper]`, `[trembling]` |
| Punctuation দিয়ে prosody | CAPS দিয়ে চেঁচানো |
| `class_temperature` 0.3–0.5 | Default greedy (flat output) |
| Segments concatenate | Single giant text block |

Run all cells sequentially in Kaggle with **GPU T4** enabled.

In [ ]:
# ⚙️ Step 1: Check GPU Status
!nvidia-smi
import torch
print(f"\n✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 📦 Step 2: Clone & Install OmniVoice Studio
!pip install -q uv
!git clone https://github.com/debpalash/OmniVoice-Studio.git /kaggle/working/omnivoice-studio
%cd /kaggle/working/omnivoice-studio
!uv sync

In [ ]:
# ⚡ Step 3: Launch OmniVoice Backend Server in Background
import os, time, subprocess
%cd /kaggle/working/omnivoice-studio

log_file = open("/tmp/omnivoice.log", "w")
subprocess.Popen(["uv", "run", "python", "backend/main.py"], stdout=log_file, stderr=log_file)

print("⏳ Waiting 15s for server startup...")
time.sleep(15)
!curl -sf http://localhost:3900/health || echo '❌ Server starting failed! Check /tmp/omnivoice.log'

In [ ]:
# =============================================================================
# 🎭 Step 4: V8 — DOCUMENTED EMOTIONAL TTS + DIRECT DOWNLOAD
# =============================================================================

import os, time, json, urllib.request, wave, base64
from IPython.display import HTML, display

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v8_documented_emotions"
API_URL = "http://localhost:3900/v1/audio/speech"

# ─── WAV Concatenation ─────────────────────────────────────────────────────
def concat_wavs(wav_paths, output_path):
    params_set = False
    all_frames = b""
    for wp in wav_paths:
        if not os.path.exists(wp): continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())
    if not params_set:
        print("  ❌ No WAV files found!")
        return
    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)
    kb = os.path.getsize(output_path) // 1024
    print(f"  🎬 FINAL: {os.path.basename(output_path)} ({kb} KB)")

# ─── Silence Generator ─────────────────────────────────────────────────────
def make_silence_wav(duration_ms, output_path, sample_rate=22050, channels=1, sampwidth=2):
    num_samples = int(sample_rate * duration_ms / 1000)
    silence = b"\x00\x00" * num_samples * channels
    with wave.open(output_path, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sampwidth)
        wf.setframerate(sample_rate)
        wf.writeframes(silence)

# ─── Segment Generator ─────────────────────────────────────────────────────
def gen_segment(text, voice="onyx", model="tts-1-hd", output_path="",
                speed=1.0, num_step=32, guidance_scale=2.0,
                class_temperature=0.0, postprocess_output=True,
                instruct=None, seed=None):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": model, "voice": voice, "input": text,
        "response_format": "wav", "speed": speed,
        "num_step": num_step, "guidance_scale": guidance_scale,
    }
    if class_temperature > 0: payload["class_temperature"] = class_temperature
    if not postprocess_output: payload["postprocess_output"] = False
    if instruct: payload["instruct"] = instruct
    if seed is not None: payload["seed"] = seed
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            dur = time.time() - t0
            kb = len(content) // 1024
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:40s} │ {kb:5d} KB │ {dur:4.1f}s")
            return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")
        return False

# ═════════════════════════════════════════════════════════════════════════════
# 📋 DOCUMENTED TAGS ONLY:
#    [laughter] [sigh] [breath] [pause] [pause 500ms] [pause 1.5s]
# ❌ DO NOT USE: [cry] [whisper] [trembling] [excited]
# ═════════════════════════════════════════════════════════════════════════════

VOICE = "onyx"
FILENAME = "v8_final"

SEGMENTS = [
    {"tag": "01_narrator_opening", "text": ("What if the greatest invention in human history... wasn't a machine that traveled through space — but one that crossed possibilities?"), "speed": 0.95, "guidance_scale": 2.2, "class_temperature": 0.3},
    {"tag": "p1", "silence_ms": 500},
    {"tag": "02_doubt_laughter", "text": ("[laughter] Another universe? Seriously? [laughter] Scientists kept working. Everyone else kept doubting."), "speed": 1.05, "guidance_scale": 2.0, "class_temperature": 0.2},
    {"tag": "p2", "silence_ms": 400},
    {"tag": "03_excitement", "text": ("And suddenly — the impossible became real! We saw dinosaurs, still walking, beneath blood-red skies! We found another Earth where humanity was born on Mars!"), "speed": 1.05, "guidance_scale": 2.5, "class_temperature": 0.4},
    {"tag": "p3", "silence_ms": 800},
    {"tag": "04_personal_shift", "text": ("But me? [pause 800ms] No. None of those worlds mattered. Not one. [pause 500ms] I was searching... for someone."), "speed": 0.85, "guidance_scale": 2.0, "class_temperature": 0.2, "postprocess_output": False},
    {"tag": "p4", "silence_ms": 600},
    {"tag": "05_grief", "text": ("Three years ago... cancer... stole my mother. [pause 1s] No warning. No mercy. No second chance."), "speed": 0.80, "guidance_scale": 2.0, "class_temperature": 0.3, "postprocess_output": False},
    {"tag": "p5", "silence_ms": 800},
    {"tag": "06_hospital", "text": ("[sigh] I watched the hospital monitor... become... silent. [pause 1.5s] I held her hand — hoping — just hoping — she'd squeeze mine one... last... time. [pause 2s] She never did."), "speed": 0.78, "guidance_scale": 2.0, "class_temperature": 0.4, "postprocess_output": False},
    {"tag": "p6", "silence_ms": 1000},
    {"tag": "07_pain", "text": ("I know she'll never answer. I know that! [pause 500ms] But I still call. [pause 800ms] Because sometimes... hope hurts more than reality."), "speed": 0.82, "guidance_scale": 2.5, "class_temperature": 0.5},
    {"tag": "p7", "silence_ms": 800},
    {"tag": "08_plea", "text": ("[breath] Listen. [pause 500ms] Take me to the universe... where my mother... never died."), "speed": 0.80, "guidance_scale": 1.5, "class_temperature": 0.2, "postprocess_output": False},
    {"tag": "p8", "silence_ms": 1200},
    {"tag": "09_warmth", "text": ("There she was. [pause 500ms] Alive. Smiling. Making breakfast. Humming the exact same song she used to sing... every Sunday morning."), "speed": 0.88, "guidance_scale": 2.2, "class_temperature": 0.3},
    {"tag": "p9", "silence_ms": 800},
    {"tag": "10_ending", "text": ("[pause 1s] She didn't know I wasn't her son. [pause 800ms] I was a broken man... borrowing someone else's miracle."), "speed": 0.82, "guidance_scale": 2.0, "class_temperature": 0.3, "postprocess_output": False},
]

# ═════════════════════════════════════════════════════════════════════════════
# 🚀 GENERATE → CONCATENATE → DOWNLOAD
# ═════════════════════════════════════════════════════════════════════════════
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "_segments")
os.makedirs(seg_dir, exist_ok=True)

speech_segs = [s for s in SEGMENTS if "text" in s]
pause_segs = [s for s in SEGMENTS if "silence_ms" in s]

print("=" * 70)
print("🎭  VERSION 8 — DOCUMENTATION-CORRECT EMOTIONAL TTS")
print("=" * 70)
print(f"🎙️  Voice     : {VOICE}")
print(f"🎬  Segments  : {len(speech_segs)} speech + {len(pause_segs)} pauses")
print(f"📂  Output    : {BASE_OUTPUT_DIR}")
print("=" * 70)
print()

wav_order = []
generated = 0

for seg in SEGMENTS:
    tag = seg["tag"]
    wav_path = os.path.join(seg_dir, f"{tag}.wav")

    if "silence_ms" in seg:
        make_silence_wav(seg["silence_ms"], wav_path)
        wav_order.append(wav_path)
        print(f"  ⏸️  {tag:40s} │ {seg['silence_ms']}ms silence")
        continue

    success = gen_segment(
        text=seg["text"], voice=VOICE, model="tts-1-hd",
        output_path=wav_path, speed=seg.get("speed", 0.88),
        num_step=32, guidance_scale=seg.get("guidance_scale", 2.0),
        class_temperature=seg.get("class_temperature", 0.0),
        postprocess_output=seg.get("postprocess_output", True),
        instruct=seg.get("instruct"), seed=seg.get("seed"),
    )
    if success:
        wav_order.append(wav_path)
        generated += 1

# ─── Concatenate ──────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("🔗 Concatenating all segments...")
final_path = os.path.join(BASE_OUTPUT_DIR, f"{FILENAME}_{VOICE}.wav")
concat_wavs(wav_order, final_path)

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"🎉  DONE! {generated} segments → 1 file")
print(f"  ⏱️  {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  📁 {final_path}")
print(f"{'=' * 70}")

# ═════════════════════════════════════════════════════════════════════════════
# 📥 DIRECT DOWNLOAD BUTTON (404-proof, base64)
# ═════════════════════════════════════════════════════════════════════════════
if os.path.exists(final_path):
    with open(final_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl_name = os.path.basename(final_path)
    html = f'''<a download="{dl_name}" href="data:audio/wav;base64,{b64}">
    <button style="padding:14px 28px; background:linear-gradient(135deg,#667eea,#764ba2);
    color:white; border:none; border-radius:8px; cursor:pointer; font-weight:bold;
    font-size:16px; margin:10px 0;">
    ⬇️ Download {dl_name}
    </button></a>'''
    print("\n⬇️  নিচের বাটনে ক্লিক করে সরাসরি ডাউনলোড করুন:")
    display(HTML(html))
else:
    print("\n❌ ফাইল তৈরি হয়নি — উপরের error চেক করুন।")